# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [5]:

EVENT_NAME = '202408_TropicalStorm_Ernesto'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'TROPICS'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 4 .tif files in the S3 bucket.


['drcs_activations/202408_TropicalStorm_Ernesto/TROPICS/Final_TC_Ernesto_Ch12_Bd5TROPICS03_BRTT_L1B_Orbit06789_V05_02_ST20240816_013724_ET20240816_031156_CT20240816_152730.tif',
 'drcs_activations/202408_TropicalStorm_Ernesto/TROPICS/Final_TC_Ernesto_Ch12_Bd5TROPICS06_BRTT_L1B_Orbit07025_V05_02_ST20240813_132029_ET20240813_145452_CT20240814_025443.tif',
 'drcs_activations/202408_TropicalStorm_Ernesto/TROPICS/Final_TC_Ernesto_Ch12_Bd5TROPICS06_BRTT_L1B_Orbit07056_V05_02_ST20240815_140612_ET20240815_154034_CT20240816_033607 (1).tif',
 'drcs_activations/202408_TropicalStorm_Ernesto/TROPICS/Final_TC_Ernesto_Ch12_Bd5TROPICS06_BRTT_L1B_Orbit07056_V05_02_ST20240815_140612_ET20240815_154034_CT20240816_033607.tif']

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [9]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 14
  - Total size: 3.98 GB

📁 Cached files (first 10):
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002653_DVR_RTC20_G_gpuned_0610_WM.tif (3.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002653_DVR_RTC20_G_gpuned_0610_rgb.tif (258.2 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002719_DVR_RTC20_G_gpuned_F141_WM.tif (2.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240430T002719_DVR_RTC20_G_gpuned_F141_rgb.tif (289.2 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240507T122323_DVR_RTC20_G_gpuned_5BA0_WM.tif (2.7 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240507T122323_DVR_RTC20_G_gpuned_5BA0_rgb.tif (321.6 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240512T002655_DVR_RTC20_G_gpuned_EC9C_WM.tif (9.1 MB)
  - drcs_activations/202405_Flood_TX/sentinel1/S1A_IW_20240512T002720_DVR_RTC20_G_gpuned_D32B_WM.tif (2

(14, 4268563394)

In [10]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [13]:
keys
keys = [i for i in keys if " (1)" not in i]

# trueColor

In [16]:
def create_cog_filename_tropics(f, EVENT_NAME):
    """Create COG filename for TROPICS files with date at the end."""
    from pathlib import Path
    import re
    
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Remove any duplicate numbering like " (1)" from the filename
    filename = re.sub(r'\s*\(\d+\)$', '', filename)
    
    # Extract components from TROPICS filename
    # Pattern to match: Final_TC_Ernesto_Ch12_Bd5TROPICS03_BRTT_L1B_Orbit06789_V05_02_ST20240816_013724_ET20240816_031156_CT20240816_152730
    pattern = r'Final_TC_(\w+)_Ch(\d+)_Bd(\d+)(TROPICS\d+)_.*_Orbit(\d+)_.*_ST(\d{8})_\d+_ET\d+_\d+_CT\d+_\d+'
    match = re.search(pattern, filename)
    
    if match:
        storm_name = match.group(1)  # Ernesto
        channel = match.group(2)      # 12
        band = match.group(3)         # 5
        satellite = match.group(4)    # TROPICS03
        orbit = match.group(5)        # 06789
        date_str = match.group(6)     # 20240816
        
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Build new filename
        cog_filename = f'{EVENT_NAME}_{satellite}_Ch{channel}_Bd{band}_{storm_name}_Orbit{orbit}_{formatted_date}_day{extension}'
    else:
        # Try a simpler pattern if the full pattern doesn't match
        st_pattern = r'ST(\d{8})'
        st_match = re.search(st_pattern, filename)
        
        if st_match:
            date_str = st_match.group(1)
            formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
            
            # Extract just the essential parts
            parts = filename.split('_')
            essential_parts = []
            
            for part in parts:
                if part.startswith('TROPICS'):
                    essential_parts.append(part)
                elif part.startswith('Ch'):
                    essential_parts.append(part)
                elif part.startswith('Bd'):
                    essential_parts.append(part)
            
            cog_filename = f'{EVENT_NAME}_{"_".join(essential_parts)}_{formatted_date}_day{extension}'
        else:
            # Fallback
            cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename

filter_str = ""

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if filter_str in f]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_tropics(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202408_TropicalStorm_Ernesto_TROPICS03_Ch12_Bd5_Ernesto_Orbit06789_2024-08-16_day.tif
  202408_TropicalStorm_Ernesto_TROPICS06_Ch12_Bd5_Ernesto_Orbit07025_2024-08-13_day.tif
  202408_TropicalStorm_Ernesto_TROPICS06_Ch12_Bd5_Ernesto_Orbit07056_2024-08-15_day.tif


In [18]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_tropics, 
                                target_dir = "Landsat/TROPICS", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202408_TropicalStorm_Ernesto_TROPICS03_Ch12_Bd5_Ernesto_Orbit06789_2024-08-16_day.tif
  202408_TropicalStorm_Ernesto_TROPICS06_Ch12_Bd5_Ernesto_Orbit07025_2024-08-13_day.tif
  202408_TropicalStorm_Ernesto_TROPICS06_Ch12_Bd5_Ernesto_Orbit07056_2024-08-15_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202408_TropicalStorm_Ernesto/TROPICS
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/TROPICS

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202408_TropicalStorm_Ernesto

[1/3] Processing: drcs_activations/202408_TropicalStorm_Ernesto/TROPICS/Final_TC_Ernesto_Ch12_Bd5TROPICS03_BRTT_L1B_Orbit06789_V05_02_ST20240816_013724_ET20240816_031156_CT20240816_152730.tif
   Output filename: 202408_TropicalStorm_Ernesto_TROPICS03_Ch12_Bd5_Ernesto_Orbit06789_2024-08-16_day.tif
   [MEMORY] Initial: 298.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJE

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=269.4626159667969, max=288.7207946777344, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpubz5ucu2_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp1_096qth.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=277.3732604980469, max=289.5521240234375, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpriy_rl4a_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp7jjc3gw9.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/TROPICS/202408_TropicalStorm_Ernesto_TROPICS06_Ch12_Bd5_Ernesto_Orbit07056_2024-08-15_day.tif
   [MEMORY] Final: 680.6 MB (Change: +66.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202408_TropicalStorm_Ernesto_TROPICS06_Ch12_Bd5_Ernesto_Orbit07056_2024-08-15_day.tif

✅ Batch processing complete: 3 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/TROPICS/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/TROPICS/files_converted.csv
📁 COGs saved locally to: output/202408_TropicalStorm_Ernesto

📊 BATCH PROCESSING SUMMARY
Total files processed: 3
Successful: 3
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-10T1

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [18]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 982.3 MB
  Available memory: 27848.8 MB
  Memory percent used: 11.9%
